<a href="https://colab.research.google.com/github/AY0ungKim/Causal_Master/blob/main/(LAB)_GEE_exercise_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Load the Libraries
### - In python, it loads a library so we can use its functions.

In [1]:
# Bring the 'Earth Engine' toolbox into this script.
import ee
# Bring the 'Time' tool to help the computer wait during tasks.
import time
# Import geemap to display interactive maps in Colab
import geemap

# 1. Configuration: Project and Data Parameters
### - Go to code editor (https://code.earthengine.google.com/) and check the project id and path for asset (shapefile)

In [2]:
project_id = 'gee-example-483611' # Your Google Cloud Project ID
folder_name = "ERA5_Monthly_Seoul" # The folder name in your Google Drive (customize)
region_asset = 'projects/gee-example-483611/assets/grid_5km_seoul' # Your uploaded shapefile (asset)

# 2. Authenticate and Initialize the Earth Engine API
### - It ensures we have the right credentials to use the earth engine service

In [13]:
try:
    # Try to initialize with the specific project ID
    ee.Initialize(project=project_id)
except Exception as e:
    # If initialization fails, run authentication workflow
    ee.Authenticate()
    ee.Initialize(project=project_id)

# 3. Define the Target Dataset and Study Area
We use the Monthly Aggregated ERA5-Land dataset which is already pre-calculated by Google

You need to refer to the website and write down the Earth Engine Snippet for the dataset you want to extract.
(ex) "ECMWF/ERA5_LAND/MONTHLY_AGGR"

- ee.ImageCollection: Accesses the ERA5-Land monthly aggregated dataset (temperature, precipitation, etc.).
- ee.FeatureCollection: Loads your uploaded grid map (Seoul 5km grid) from Earth Engine assets.

In [4]:
# Load the 'ERA5-Land Monthly' dataset (a huge collection of climate images)
dataset = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

#Load your 'Shapefile' (the 5km grid map of Seoul) from your Assets
region = ee.FeatureCollection(region_asset)

You can visualize the study area using "geemap" library

- geemap.Map(): Creates an interactive map interface for Jupyter/Colab notebooks.
-  google_map_en: A URL template for Google Maps tiles.
- 'lyrs=m' stands for standard roadmap, and 'hl=en' forces the labels to be in English.
- Map.add_tile_layer: Injects the Google Roadmap into the geemap instance.
- Map.centerObject: Automatically moves the camera to the center of your 'region' (Seoul).

In [5]:
Map = geemap.Map()

# Add Google Roadmap with English Labels
google_map_en = "https://mt1.google.com/vt/lyrs=m&hl=en&x={x}&y={y}&z={z}"
Map.add_tile_layer(google_map_en, name="Google Roadmap (English)", attribution="Google")

# Add layers over the English Map
Map.addLayer(region.style(color='black', fillColor='00000000'), {}, 'Grid Outline')

# Center the map and Display
Map.centerObject(region, 11)
Map

Map(center=[37.55190188565766, 126.99180675359528], controls=(WidgetControl(options=['position', 'transparent_…

# 4. Define the Bands to Extract
### - You need to refer to the website and write down the band names for the dataset you want to extract.
(ex) https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY_AGGR
### - For the next lecture, we will extract the following variables
1. dewpoint_temperature_2m
2. total_precipitation_sum
3. temperature_2m
4. surface_pressure
5. u_component_of_wind_10m
6. v_component_of_wind_10m
7. soil_temperature_level_1
8. total_evaporation_sum
9. leaf_area_index_high_vegetation
10. leaf_area_index_low_vegetation

In [6]:
# Create a list of specific weather variables (Bands) we want to extract
target_bands = [
    "dewpoint_temperature_2m",
    "total_precipitation_sum",
    "temperature_2m",
    "surface_pressure",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "soil_temperature_level_1",
    "total_evaporation_sum",
    "leaf_area_index_high_vegetation",
    "leaf_area_index_low_vegetation"
]

# 5. Filter the Collection by Date
### - We will set the period from **2015 to 2019**
#### The end date is exclusive, so '2020-01-01' includes up to December 2019

In [7]:
filtered_coll = dataset.filterDate('2015-01-01', '2020-01-01').select(target_bands)

- .reduceRegions(): Overlays your shapefile (region) on the satellite image.
- reducer=ee.Reducer.mean(): Specifies that we want the average value of all pixels within each grid cell.
- scale: The spatial resolution for the calculation

- image.date(): Retrieves the sensing time (timestamp) of the satellite image.
- .get(): Extracts individual components like year, month
- stats.map(lambda f: f.set(...)): Since 'stats' is a collection of many grids,
> we loop through each grid (feature 'f') and add the year and month as new attributes.
> This ensures that when you open the CSV, every row knows exactly which date it belongs to.

In [8]:
def extract_monthly_stats(image):
    # 'reduceRegions' calculates the mean pixel value for each grid in your shapefile
    stats = image.reduceRegions(
        collection=region,
        reducer=ee.Reducer.mean(),
        scale=1000
    )

    # Get the year and month from the image timestamp to add to each row in the CSV
    year = image.date().get('year')
    month = image.date().get('month')

    # Map a function to set 'year' and 'month' properties for every feature (grid)
    return stats.map(lambda f: f.set({
        'year': year,
        'month': month
    }))

# 7. Apply the Function and Flatten the Result
### Resulting 'final_features' will be a table where each row is (Grid ID + Date + Bands)

In [9]:
final_features = filtered_coll.map(extract_monthly_stats).flatten()

# 8. Define Export Selectors (Columns for your CSV)
# These names must match the property names in your Asset and the Band names
### - For Single-Band Images (e.g., only EVI)
> When you apply a reducer to a single band, GEE automatically renames the resulting output property to the name of the reducer (e.g., "mean").


### - For Multi-Band Images (2 or more bands)
> When processing multiple bands simultaneously, GEE preserves the original band names (often appending the reducer name, like "temperature_2m", "dewpoint_temperature_2m") to keep them distinct.


In [10]:
csv_columns = ['grid_id', 'year', 'month'] + target_bands

# 9. Start the Export Task to Google Drive

In [14]:
description = "ERA5_Seoul_Monthly_2015_2019"
task = ee.batch.Export.table.toDrive(
    collection=final_features,
    description=description,
    folder=folder_name,
    fileNamePrefix=description,
    fileFormat="CSV",
    selectors=csv_columns # This ensures the column order matches your Excel image
)

task.start()
print(f"Export started! Please check your GEE Tasks or Google Drive folder: {folder_name}")

Export started! Please check your GEE Tasks or Google Drive folder: ERA5_Monthly_Seoul
